In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("LocalSparkTest")
    .master("local[1]")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.ui.enabled", "false")
    .config("spark.sql.shuffle.partitions", "1")
    .getOrCreate()
)

26/06/13 08:20:36 WARN Utils: Your hostname, MKHOMBP14018.local resolves to a loopback address: 127.0.0.1; using 10.10.10.172 instead (on interface en0)
26/06/13 08:20:36 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/13 08:20:36 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
#18.Calculate the percentage of total revenue contributed by each product category.


In [5]:
from pyspark.sql.functions import col

data = [
    (1, "Electronics", "Laptop", 1000),
    (2, "Electronics", "Mobile", 800),
    (3, "Electronics", "Tablet", 700),
    (4, "Furniture", "Chair", 300),
    (5, "Furniture", "Table", 500),
    (6, "Clothing", "Shirt", 200),
    (7, "Clothing", "Jeans", 400),
    (8, "Grocery", "Rice", 100),
    (9, "Grocery", "Oil", 200)
]

columns = ["sale_id", "category", "product", "revenue"]

sales_df = spark.createDataFrame(data, columns)

sales_df.createOrReplaceTempView("sales")

sales_df.show()

+-------+-----------+-------+-------+
|sale_id|   category|product|revenue|
+-------+-----------+-------+-------+
|      1|Electronics| Laptop|   1000|
|      2|Electronics| Mobile|    800|
|      3|Electronics| Tablet|    700|
|      4|  Furniture|  Chair|    300|
|      5|  Furniture|  Table|    500|
|      6|   Clothing|  Shirt|    200|
|      7|   Clothing|  Jeans|    400|
|      8|    Grocery|   Rice|    100|
|      9|    Grocery|    Oil|    200|
+-------+-----------+-------+-------+



In [12]:
def runsql(query):
    spark.sql(query).show()
    

In [17]:
runsql(
    """with category_sales as (
    select category, SUM(revenue) as revenue_cat from sales group by category),
    total_sales_data  as (
    select  SUM(revenue) as total_sales from sales )
    Select category, 100*revenue_cat/total_sales_data.total_sales from category_sales, total_sales_data
    """
)

+-----------+-----------------------------------+
|   category|((100 * revenue_cat) / total_sales)|
+-----------+-----------------------------------+
|Electronics|                 59.523809523809526|
|  Furniture|                 19.047619047619047|
|   Clothing|                 14.285714285714286|
|    Grocery|                  7.142857142857143|
+-----------+-----------------------------------+



In [25]:
runsql(
    """with category_sales as (
    select category, SUM(revenue) as revenue_cat from sales group by category)

    Select category, revenue_cat, Sum(revenue_cat) over (partition by category ),100*revenue_cat/Sum(revenue_cat) over ( ) as_cat from category_sales
    """
)

+-----------+-----------+------------------------------------------------------------------------------------------------------+------------------+
|   category|revenue_cat|sum(revenue_cat) OVER (PARTITION BY category ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING)|            as_cat|
+-----------+-----------+------------------------------------------------------------------------------------------------------+------------------+
|   Clothing|        600|                                                                                                   600|14.285714285714286|
|Electronics|       2500|                                                                                                  2500|59.523809523809526|
|  Furniture|        800|                                                                                                   800|19.047619047619047|
|    Grocery|        300|                                                                                       

26/06/13 08:30:48 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 08:30:48 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 08:30:48 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 08:30:48 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


In [27]:
result_df = spark.sql("""
WITH category_revenue AS (
    SELECT
        category,
        SUM(revenue) AS category_revenue
    FROM sales
    GROUP BY category
),

total_revenue AS (
    SELECT
        SUM(category_revenue) AS total_revenue
    FROM category_revenue
)

SELECT
    c.category,
    c.category_revenue,
    t.total_revenue,
    ROUND(
        (c.category_revenue / t.total_revenue) * 100,
        2
    ) AS revenue_percentage
FROM category_revenue c
CROSS JOIN total_revenue t
ORDER BY revenue_percentage DESC
""")

result_df.show()

+-----------+----------------+-------------+------------------+
|   category|category_revenue|total_revenue|revenue_percentage|
+-----------+----------------+-------------+------------------+
|Electronics|            2500|         4200|             59.52|
|  Furniture|             800|         4200|             19.05|
|   Clothing|             600|         4200|             14.29|
|    Grocery|             300|         4200|              7.14|
+-----------+----------------+-------------+------------------+



In [29]:
from pyspark.sql.functions import sum as spark_sum, round
from pyspark.sql.window import Window
category_revenue_df = (
    sales_df
    .groupBy("category")
    .agg(spark_sum("revenue").alias("category_revenue"))
)

result_df = (
    category_revenue_df
    .withColumn(
        "total_revenue",
        spark_sum("category_revenue").over(Window.partitionBy())
    )
    .withColumn(
        "revenue_percentage",
        round((col("category_revenue") / col("total_revenue")) * 100, 2)
    )
    .orderBy(col("revenue_percentage").desc())
)

result_df.show()

+-----------+----------------+-------------+------------------+
|   category|category_revenue|total_revenue|revenue_percentage|
+-----------+----------------+-------------+------------------+
|Electronics|            2500|         4200|             59.52|
|  Furniture|             800|         4200|             19.05|
|   Clothing|             600|         4200|             14.29|
|    Grocery|             300|         4200|              7.14|
+-----------+----------------+-------------+------------------+



26/06/13 08:31:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 08:31:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 08:31:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 08:31:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 08:31:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


In [30]:
#19.What % of users who signed up in a given month made a purchase within 30 days? (conversion)

In [31]:
from pyspark.sql.functions import col, to_date

users_data = [
    (1, "Alice", "2026-01-05"),
    (2, "Bob", "2026-01-10"),
    (3, "Charlie", "2026-01-20"),
    (4, "David", "2026-02-03"),
    (5, "Eva", "2026-02-15"),
    (6, "Frank", "2026-02-22"),
    (7, "Grace", "2026-03-01")
]

users_columns = ["user_id", "user_name", "signup_date"]

users_df = spark.createDataFrame(users_data, users_columns)

users_df = users_df.withColumn(
    "signup_date",
    to_date(col("signup_date"))
)

purchases_data = [
    (101, 1, "2026-01-15", 500),  # Alice: within 30 days
    (102, 2, "2026-02-20", 300),  # Bob: after 30 days
    (103, 3, "2026-02-05", 700),  # Charlie: within 30 days
    (104, 4, "2026-02-20", 200),  # David: within 30 days
    (105, 5, "2026-04-01", 400),  # Eva: after 30 days
    (106, 7, "2026-03-10", 900)   # Grace: within 30 days
]

purchases_columns = ["purchase_id", "user_id", "purchase_date", "amount"]

purchases_df = spark.createDataFrame(purchases_data, purchases_columns)

purchases_df = purchases_df.withColumn(
    "purchase_date",
    to_date(col("purchase_date"))
)

users_df.createOrReplaceTempView("users")
purchases_df.createOrReplaceTempView("purchases")

users_df.show()
purchases_df.show()

+-------+---------+-----------+
|user_id|user_name|signup_date|
+-------+---------+-----------+
|      1|    Alice| 2026-01-05|
|      2|      Bob| 2026-01-10|
|      3|  Charlie| 2026-01-20|
|      4|    David| 2026-02-03|
|      5|      Eva| 2026-02-15|
|      6|    Frank| 2026-02-22|
|      7|    Grace| 2026-03-01|
+-------+---------+-----------+

+-----------+-------+-------------+------+
|purchase_id|user_id|purchase_date|amount|
+-----------+-------+-------------+------+
|        101|      1|   2026-01-15|   500|
|        102|      2|   2026-02-20|   300|
|        103|      3|   2026-02-05|   700|
|        104|      4|   2026-02-20|   200|
|        105|      5|   2026-04-01|   400|
|        106|      7|   2026-03-10|   900|
+-----------+-------+-------------+------+



In [44]:
runsql(
"""
With purchage_user_history as (
select p.user_id ,user_name, signup_date, purchase_date,amount,  date_diff(purchase_date,signup_date) sing_pur from users as u 
INNER JOIN purchases p 
ON u.user_id=p.user_id
)
select user_id,sing_pur,amount,purchase_date from purchage_user_history where sing_pur<=30
"""

    
)

+-------+--------+------+-------------+
|user_id|sing_pur|amount|purchase_date|
+-------+--------+------+-------------+
|      1|      10|   500|   2026-01-15|
|      3|      16|   700|   2026-02-05|
|      4|      17|   200|   2026-02-20|
|      7|       9|   900|   2026-03-10|
+-------+--------+------+-------------+



In [45]:
conversion_df = spark.sql("""
SELECT
    DATE_FORMAT(u.signup_date, 'yyyy-MM') AS signup_month,
    COUNT(DISTINCT u.user_id) AS total_signup_users,
    COUNT(DISTINCT CASE
        WHEN p.purchase_date >= u.signup_date
         AND p.purchase_date <= DATE_ADD(u.signup_date, 30)
        THEN u.user_id
    END) AS converted_users,
    ROUND(
        COUNT(DISTINCT CASE
            WHEN p.purchase_date >= u.signup_date
             AND p.purchase_date <= DATE_ADD(u.signup_date, 30)
            THEN u.user_id
        END)
        / COUNT(DISTINCT u.user_id) * 100,
        2
    ) AS conversion_rate_percent
FROM users u
LEFT JOIN purchases p
    ON u.user_id = p.user_id
GROUP BY DATE_FORMAT(u.signup_date, 'yyyy-MM')
ORDER BY signup_month
""")

conversion_df.show()

+------------+------------------+---------------+-----------------------+
|signup_month|total_signup_users|converted_users|conversion_rate_percent|
+------------+------------------+---------------+-----------------------+
|     2026-01|                 3|              2|                  66.67|
|     2026-02|                 3|              1|                  33.33|
|     2026-03|                 1|              1|                  100.0|
+------------+------------------+---------------+-----------------------+



In [46]:
conversion_df = spark.sql("""
WITH signup_cohort AS (
    SELECT
        user_id,
        user_name,
        signup_date,
        DATE_FORMAT(signup_date, 'yyyy-MM') AS signup_month
    FROM users
),

purchase_within_30_days AS (
    SELECT DISTINCT
        u.user_id
    FROM signup_cohort u
    JOIN purchases p
        ON u.user_id = p.user_id
       AND p.purchase_date >= u.signup_date
       AND p.purchase_date <= DATE_ADD(u.signup_date, 30)
),

cohort_conversion AS (
    SELECT
        u.signup_month,
        COUNT(DISTINCT u.user_id) AS total_signup_users,
        COUNT(DISTINCT p.user_id) AS converted_users
    FROM signup_cohort u
    LEFT JOIN purchase_within_30_days p
        ON u.user_id = p.user_id
    GROUP BY u.signup_month
)

SELECT
    signup_month,
    total_signup_users,
    converted_users,
    ROUND(
        converted_users / total_signup_users * 100,
        2
    ) AS conversion_rate_percent
FROM cohort_conversion
ORDER BY signup_month
""")

conversion_df.show()

+------------+------------------+---------------+-----------------------+
|signup_month|total_signup_users|converted_users|conversion_rate_percent|
+------------+------------------+---------------+-----------------------+
|     2026-01|                 3|              2|                  66.67|
|     2026-02|                 3|              1|                  33.33|
|     2026-03|                 1|              1|                  100.0|
+------------+------------------+---------------+-----------------------+



In [47]:
from pyspark.sql.functions import date_format, date_add, countDistinct, when, round

joined_df = (
    users_df.alias("u")
    .join(
        purchases_df.alias("p"),
        on="user_id",
        how="left"
    )
    .withColumn("signup_month", date_format(col("signup_date"), "yyyy-MM"))
    .withColumn(
        "converted_flag",
        when(
            (col("p.purchase_date") >= col("u.signup_date")) &
            (col("p.purchase_date") <= date_add(col("u.signup_date"), 30)),
            col("user_id")
        )
    )
)

result_df = (
    joined_df
    .groupBy("signup_month")
    .agg(
        countDistinct("user_id").alias("total_signup_users"),
        countDistinct("converted_flag").alias("converted_users")
    )
    .withColumn(
        "conversion_rate_percent",
        round((col("converted_users") / col("total_signup_users")) * 100, 2)
    )
    .orderBy("signup_month")
)

result_df.show()

+------------+------------------+---------------+-----------------------+
|signup_month|total_signup_users|converted_users|conversion_rate_percent|
+------------+------------------+---------------+-----------------------+
|     2026-01|                 3|              2|                  66.67|
|     2026-02|                 3|              1|                  33.33|
|     2026-03|                 1|              1|                  100.0|
+------------+------------------+---------------+-----------------------+



In [48]:
#20 Write a query to get the distribution of the number of conversations created by each user per day in a given year (reported Google question — group by user+day, then group by the count).


In [49]:
from pyspark.sql.functions import to_timestamp, col

data = [
    (101, 1, "2026-01-01 09:00:00"),
    (102, 1, "2026-01-01 10:00:00"),
    (103, 1, "2026-01-02 11:00:00"),

    (104, 2, "2026-01-01 08:00:00"),
    (105, 2, "2026-01-01 09:00:00"),
    (106, 2, "2026-01-01 10:00:00"),

    (107, 3, "2026-01-01 12:00:00"),

    (108, 4, "2026-02-05 13:00:00"),
    (109, 4, "2026-02-05 14:00:00"),

    (110, 5, "2025-12-31 23:00:00")  # outside target year
]

columns = ["conversation_id", "user_id", "created_at"]

conversations_df = spark.createDataFrame(data, columns)

conversations_df = conversations_df.withColumn(
    "created_at",
    to_timestamp(col("created_at"))
)

conversations_df.createOrReplaceTempView("conversations")

result_df = spark.sql("""
WITH user_daily_conversation_count AS (
    SELECT
        user_id,
        DATE(created_at) AS conversation_date,
        COUNT(*) AS conversations_created
    FROM conversations
    WHERE created_at >= DATE '2026-01-01'
      AND created_at <  DATE '2027-01-01'
    GROUP BY
        user_id,
        DATE(created_at)
)

SELECT
    conversations_created,
    COUNT(*) AS user_day_count
FROM user_daily_conversation_count
GROUP BY conversations_created
ORDER BY conversations_created
""")

result_df.show()

+---------------------+--------------+
|conversations_created|user_day_count|
+---------------------+--------------+
|                    1|             2|
|                    2|             2|
|                    3|             1|
+---------------------+--------------+



In [51]:
runsql(""" 
select * from conversations
"""
)

+---------------+-------+-------------------+
|conversation_id|user_id|         created_at|
+---------------+-------+-------------------+
|            101|      1|2026-01-01 09:00:00|
|            102|      1|2026-01-01 10:00:00|
|            103|      1|2026-01-02 11:00:00|
|            104|      2|2026-01-01 08:00:00|
|            105|      2|2026-01-01 09:00:00|
|            106|      2|2026-01-01 10:00:00|
|            107|      3|2026-01-01 12:00:00|
|            108|      4|2026-02-05 13:00:00|
|            109|      4|2026-02-05 14:00:00|
|            110|      5|2025-12-31 23:00:00|
+---------------+-------+-------------------+



In [57]:
runsql(""" 
select user_id,  date(created_at) from conversations
"""
)

+-------+----------+
|user_id|created_at|
+-------+----------+
|      1|2026-01-01|
|      1|2026-01-01|
|      1|2026-01-02|
|      2|2026-01-01|
|      2|2026-01-01|
|      2|2026-01-01|
|      3|2026-01-01|
|      4|2026-02-05|
|      4|2026-02-05|
|      5|2025-12-31|
+-------+----------+



In [63]:
runsql(""" 
select user_id,  date(created_at) ,COUNT(*) date_and_User_conver_cnt from conversations group by user_id,  date(created_at)
"""
)

+-------+----------+------------------------+
|user_id|created_at|date_and_User_conver_cnt|
+-------+----------+------------------------+
|      1|2026-01-01|                       2|
|      1|2026-01-02|                       1|
|      2|2026-01-01|                       3|
|      3|2026-01-01|                       1|
|      4|2026-02-05|                       2|
|      5|2025-12-31|                       1|
+-------+----------+------------------------+



In [65]:
runsql(""" 

With user_date_conversation as (
select user_id,  date(created_at) ,COUNT(*)as date_and_User_conver_cnt  from conversations group by user_id,  date(created_at)
) 
select user_id, COUNT(user_id) user_converstion_cnt_day , sum(date_and_User_conver_cnt) total_conv from user_date_conversation group by user_id
"""
)

+-------+------------------------+----------+
|user_id|user_converstion_cnt_day|total_conv|
+-------+------------------------+----------+
|      1|                       2|         3|
|      2|                       1|         3|
|      3|                       1|         1|
|      4|                       1|         2|
|      5|                       1|         1|
+-------+------------------------+----------+



In [67]:
runsql(""" 
WITH user_day AS (

    SELECT

        user_id,

        DATE(created_at) AS day,

        COUNT(*) AS conversation_count

    FROM conversations

    WHERE created_at >= DATE '2026-01-01'

      AND created_at <  DATE '2027-01-01'

    GROUP BY user_id, DATE(created_at)

)

SELECT

    conversation_count,

    COUNT(*) AS frequency

FROM user_day

GROUP BY conversation_count;


"""
)

+------------------+---------+
|conversation_count|frequency|
+------------------+---------+
|                 2|        2|
|                 1|        2|
|                 3|        1|
+------------------+---------+



In [68]:
#21 Pivot rows to columns using conditional aggregation (COUNT(CASE WHEN ...)).


In [69]:
from pyspark.sql.functions import col, to_date

data = [
    (1, "East", "Completed", "2026-01-01"),
    (2, "East", "Completed", "2026-01-02"),
    (3, "East", "Cancelled", "2026-01-03"),
    (4, "West", "Completed", "2026-01-01"),
    (5, "West", "Pending", "2026-01-04"),
    (6, "West", "Pending", "2026-01-05"),
    (7, "North", "Cancelled", "2026-01-06"),
    (8, "North", "Completed", "2026-01-07"),
    (9, "South", "Pending", "2026-01-08")
]

columns = ["order_id", "region", "status", "order_date"]

orders_df = spark.createDataFrame(data, columns)

orders_df = orders_df.withColumn("order_date", to_date(col("order_date")))

orders_df.createOrReplaceTempView("orders")

orders_df.show()

+--------+------+---------+----------+
|order_id|region|   status|order_date|
+--------+------+---------+----------+
|       1|  East|Completed|2026-01-01|
|       2|  East|Completed|2026-01-02|
|       3|  East|Cancelled|2026-01-03|
|       4|  West|Completed|2026-01-01|
|       5|  West|  Pending|2026-01-04|
|       6|  West|  Pending|2026-01-05|
|       7| North|Cancelled|2026-01-06|
|       8| North|Completed|2026-01-07|
|       9| South|  Pending|2026-01-08|
+--------+------+---------+----------+



In [75]:
runsql(
"""
select region, COUNT(order_id) as order_id , COUNT(case when status='Completed' then order_id ELSE NULL END ) as Completed
FROM orders GROUP BY region

"""

    
)

+------+--------+---------+
|region|order_id|Completed|
+------+--------+---------+
|  East|       3|        2|
|  West|       3|        1|
| North|       2|        1|
| South|       1|        0|
+------+--------+---------+



In [76]:
result_df = spark.sql("""
SELECT
    region,

    COUNT(CASE WHEN status = 'Completed' THEN 1 END) AS completed_orders,
    COUNT(CASE WHEN status = 'Pending' THEN 1 END) AS pending_orders,
    COUNT(CASE WHEN status = 'Cancelled' THEN 1 END) AS cancelled_orders,

    COUNT(*) AS total_orders

FROM orders
GROUP BY region
ORDER BY region
""")

result_df.show()

+------+----------------+--------------+----------------+------------+
|region|completed_orders|pending_orders|cancelled_orders|total_orders|
+------+----------------+--------------+----------------+------------+
|  East|               2|             0|               1|           3|
| North|               1|             0|               1|           2|
| South|               0|             1|               0|           1|
|  West|               1|             2|               0|           3|
+------+----------------+--------------+----------------+------------+



In [77]:
result_df = spark.sql("""
SELECT
    region,

    COUNT(CASE WHEN status = 'Completed' THEN 1 END) AS completed_orders,
    COUNT(CASE WHEN status = 'Pending' THEN 1 END) AS pending_orders,
    COUNT(CASE WHEN status = 'Cancelled' THEN 1 END) AS cancelled_orders,

    COUNT(*) AS total_orders

FROM orders
GROUP BY region
ORDER BY region
""")

result_df.show()

+------+----------------+--------------+----------------+------------+
|region|completed_orders|pending_orders|cancelled_orders|total_orders|
+------+----------------+--------------+----------------+------------+
|  East|               2|             0|               1|           3|
| North|               1|             0|               1|           2|
| South|               0|             1|               0|           1|
|  West|               1|             2|               0|           3|
+------+----------------+--------------+----------------+------------+



In [79]:
from pyspark.sql.functions import count, when

result_df = (
    orders_df
    .groupBy("region")
    .agg(
        count(when(col("status") == "Completed", 1)).alias("completed_orders"),
        count(when(col("status") == "Pending", 1)).alias("pending_orders"),
        count(when(col("status") == "Cancelled", 1)).alias("cancelled_orders"),
        count("*").alias("total_orders")
    )
    .orderBy("region")
)

result_df.show()

+------+----------------+--------------+----------------+------------+
|region|completed_orders|pending_orders|cancelled_orders|total_orders|
+------+----------------+--------------+----------------+------------+
|  East|               2|             0|               1|           3|
| North|               1|             0|               1|           2|
| South|               0|             1|               0|           1|
|  West|               1|             2|               0|           3|
+------+----------------+--------------+----------------+------------+



In [80]:
#22.Compute click-through rate (clicks/impressions) per ad, handling division by zero.


In [81]:
data = [
    (1, "Ad_A", 1000, 50),
    (2, "Ad_B", 2000, 120),
    (3, "Ad_C", 0, 10),       # division by zero case
    (4, "Ad_D", 500, 0),
    (5, "Ad_E", 0, 0)         # division by zero case
]

columns = ["ad_id", "ad_name", "impressions", "clicks"]

ads_df = spark.createDataFrame(data, columns)

ads_df.createOrReplaceTempView("ads")

ads_df.show()

+-----+-------+-----------+------+
|ad_id|ad_name|impressions|clicks|
+-----+-------+-----------+------+
|    1|   Ad_A|       1000|    50|
|    2|   Ad_B|       2000|   120|
|    3|   Ad_C|          0|    10|
|    4|   Ad_D|        500|     0|
|    5|   Ad_E|          0|     0|
+-----+-------+-----------+------+



In [82]:
result_df = spark.sql("""
SELECT
    ad_id,
    ad_name,
    impressions,
    clicks,
    CASE
        WHEN impressions = 0 THEN NULL
        ELSE ROUND(clicks / impressions * 100, 2)
    END AS ctr_percent
FROM ads
ORDER BY ad_id
""")

result_df.show()

+-----+-------+-----------+------+-----------+
|ad_id|ad_name|impressions|clicks|ctr_percent|
+-----+-------+-----------+------+-----------+
|    1|   Ad_A|       1000|    50|        5.0|
|    2|   Ad_B|       2000|   120|        6.0|
|    3|   Ad_C|          0|    10|       NULL|
|    4|   Ad_D|        500|     0|        0.0|
|    5|   Ad_E|          0|     0|       NULL|
+-----+-------+-----------+------+-----------+



In [83]:
result_df = spark.sql("""
SELECT
    ad_id,
    ad_name,
    impressions,
    clicks,
    CASE
        WHEN impressions = 0 THEN NULL
        ELSE ROUND(clicks / impressions * 100, 2)
    END AS ctr_percent
FROM ads
ORDER BY ad_id
""")

result_df.show()

+-----+-------+-----------+------+-----------+
|ad_id|ad_name|impressions|clicks|ctr_percent|
+-----+-------+-----------+------+-----------+
|    1|   Ad_A|       1000|    50|        5.0|
|    2|   Ad_B|       2000|   120|        6.0|
|    3|   Ad_C|          0|    10|       NULL|
|    4|   Ad_D|        500|     0|        0.0|
|    5|   Ad_E|          0|     0|       NULL|
+-----+-------+-----------+------+-----------+



In [84]:
result_df = spark.sql("""
SELECT
    ad_id,
    ad_name,
    impressions,
    clicks,
    CASE
        WHEN impressions = 0 THEN NULL
        ELSE ROUND(clicks / impressions * 100, 2)
    END AS ctr_percent
FROM ads
ORDER BY ad_id
""")

result_df.show()

+-----+-------+-----------+------+-----------+
|ad_id|ad_name|impressions|clicks|ctr_percent|
+-----+-------+-----------+------+-----------+
|    1|   Ad_A|       1000|    50|        5.0|
|    2|   Ad_B|       2000|   120|        6.0|
|    3|   Ad_C|          0|    10|       NULL|
|    4|   Ad_D|        500|     0|        0.0|
|    5|   Ad_E|          0|     0|       NULL|
+-----+-------+-----------+------+-----------+



In [85]:
from pyspark.sql.functions import col, when, round

result_df = (
    ads_df
    .withColumn(
        "ctr_percent",
        when(col("impressions") == 0, None)
        .otherwise(round((col("clicks") / col("impressions")) * 100, 2))
    )
    .orderBy("ad_id")
)

result_df.show()

+-----+-------+-----------+------+-----------+
|ad_id|ad_name|impressions|clicks|ctr_percent|
+-----+-------+-----------+------+-----------+
|    1|   Ad_A|       1000|    50|        5.0|
|    2|   Ad_B|       2000|   120|        6.0|
|    3|   Ad_C|          0|    10|       NULL|
|    4|   Ad_D|        500|     0|        0.0|
|    5|   Ad_E|          0|     0|       NULL|
+-----+-------+-----------+------+-----------+

